In [31]:
import sqlite3
import pandas as pd

In [33]:
# Create and connect to the database
conn = sqlite3.connect('bookstore.db')
cursor = conn.cursor()

# Task 1: Database Design and Table Creation

In [41]:
# 1. Create Books Table
cursor.execute('''
CREATE TABLE Books (
book_id INTEGER PRIMARY KEY AUTOINCREMENT,
title TEXT NOT NULL,
author TEXT NOT NULL,
price REAL NOT NULL,
stock_quantity INTEGER DEFAULT 0
)
''')

In [9]:
# 2. create Customers Table
cursor.execute('''
CREATE TABLE IF NOT EXISTS Customers (
    customer_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL,
    city TEXT,
    join_date TEXT
)
''')

In [11]:
# 3. Create Orders Table
cursor.execute('''
CREATE TABLE IF NOT EXISTS Orders (
    order_id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id INTEGER,
    book_id INTEGER,
    quantity INTEGER NOT NULL,
    order_date TEXT NOT NULL,
    total_amount REAL,
    FOREIGN KEY (customer_id) REFERENCES Customers (customer_id),
    FOREIGN KEY (book_id) REFERENCES Books (book_id)
)
''')

In [13]:
print("✓ Database connection established")
print("✓ Books, Customers, and Orders tables created successfully")

✓ Database connection established
✓ Books, Customers, and Orders tables created successfully


In [43]:
print("\nSchema for books table")
print(pd.read_sql("PRAGMA table_info(books)", conn))

print("\nSchema for customers table")
print(pd.read_sql("PRAGMA table_info(customers)", conn))

print("\nSchema for orders table")
print(pd.read_sql("PRAGMA table_info(orders)", conn))


Schema for books table
   cid            name     type  notnull dflt_value  pk
0    0         book_id  INTEGER        0       None   1
1    1           title     TEXT        1       None   0
2    2          author     TEXT        1       None   0
3    3           price     REAL        1       None   0
4    4  stock_quantity  INTEGER        0          0   0

Schema for customers table
   cid         name     type  notnull dflt_value  pk
0    0  customer_id  INTEGER        0       None   1
1    1         name     TEXT        1       None   0
2    2        email     TEXT        1       None   0
3    3         city     TEXT        0       None   0
4    4    join_date     TEXT        0       None   0

Schema for orders table
   cid          name     type  notnull dflt_value  pk
0    0      order_id  INTEGER        0       None   1
1    1   customer_id  INTEGER        0       None   0
2    2       book_id  INTEGER        0       None   0
3    3      quantity  INTEGER        1       None   0

# Task 2: Data Insertion and Querying 

# Part A - Insert Data

In [45]:
# Data Lists
books_data = [('Python Programming', 'John Smith', 599.99, 25),
              ('Data Science Handbook', 'Jane Doe', 899.50, 15),
              ('Machine Learning Basics', 'Alan Turing', 1299.00, 10),
              ('SQL Essentials', 'Edgar Codd', 499.99, 30),
              ('Web Development', 'Tim Berners', 799.00, 20)]

customers_data = [('Rahul Sharma', 'rahul@email.com', 'Mumbai', '2024-01-15'),
                  ('Priya Patel', 'priya@email.com', 'Delhi', '2024-01-20'),
                  ('Amit Kumar', 'amit@email.com', 'Bangalore', '2024-02-01'),
                  ('Sneha Reddy', 'sneha@email.com', 'Hyderabad', '2024-02-10'),
                  ('Vikram Singh', 'vikram@email.com', 'Mumbai', '2024-02-15')]

In [47]:
orders_data = [
    (1, 1, 2, '2024-03-01', 1199.00),
    (1, 2, 1, '2024-03-02', 899.50),
    (2, 1, 1, '2024-03-03', 599.99),
    (2, 3, 1, '2024-03-05', 1299.00),
    (3, 4, 3, '2024-03-07', 1499.97),
    (4, 2, 1, '2024-03-10', 899.50),
    (5, 5, 2, '2024-03-12', 1598.00)
]

In [49]:
cursor.executemany('''
INSERT INTO Books (title, author, price, stock_quantity) VALUES (?,?,?,?)
''', books_data)

In [51]:
cursor.executemany(
    "INSERT INTO customers (name, email, city, join_date) VALUES (?, ?, ?, ?)",
    customers_data
)

In [53]:
cursor.executemany(
    "INSERT INTO orders (customer_id, book_id, quantity, order_date, total_amount) VALUES (?, ?, ?, ?, ?)",
    orders_data
)

In [55]:
conn.commit()

In [57]:
print("✓ Inserted books, customers and orders")

✓ Inserted books, customers and orders


In [59]:
print("\nAll Books")
print(pd.read_sql("SELECT * FROM books", conn))


All Books
   book_id                    title       author    price  stock_quantity
0        1       Python Programming   John Smith   599.99              25
1        2    Data Science Handbook     Jane Doe   899.50              15
2        3  Machine Learning Basics  Alan Turing  1299.00              10
3        4           SQL Essentials   Edgar Codd   499.99              30
4        5          Web Development  Tim Berners   799.00              20


In [61]:
print("\nAll Customers")
print(pd.read_sql("SELECT * FROM customers", conn))


All Customers
   customer_id          name             email       city   join_date
0            1  Rahul Sharma   rahul@email.com     Mumbai  2024-01-15
1            2   Priya Patel   priya@email.com      Delhi  2024-01-20
2            3    Amit Kumar    amit@email.com  Bangalore  2024-02-01
3            4   Sneha Reddy   sneha@email.com  Hyderabad  2024-02-10
4            5  Vikram Singh  vikram@email.com     Mumbai  2024-02-15


In [63]:
print("\nAll Orders")
print(pd.read_sql("SELECT * FROM orders", conn))


All Orders
   order_id  customer_id  book_id  quantity  order_date  total_amount
0         1            1        1         2  2024-03-01       1199.00
1         2            1        2         1  2024-03-02        899.50
2         3            2        1         1  2024-03-03        599.99
3         4            2        3         1  2024-03-05       1299.00
4         5            3        4         3  2024-03-07       1499.97
5         6            4        2         1  2024-03-10        899.50
6         7            5        5         2  2024-03-12       1598.00


# Part B: Complex Queries

In [65]:
query = "SELECT * FROM customers WHERE city='Mumbai'"
print(pd.read_sql(query, conn))

   customer_id          name             email    city   join_date
0            1  Rahul Sharma   rahul@email.com  Mumbai  2024-01-15
1            5  Vikram Singh  vikram@email.com  Mumbai  2024-02-15


In [67]:
query = """
SELECT book_id, title, price, stock_quantity
FROM books
WHERE price > 800 AND stock_quantity > 10
"""

print(pd.read_sql(query, conn))

   book_id                  title  price  stock_quantity
0        2  Data Science Handbook  899.5              15


In [69]:
query = "SELECT COUNT(*) AS total_orders FROM orders"
print(pd.read_sql(query, conn))

   total_orders
0             7


In [73]:
query = """ SELECT c.name, count(o.order_id) AS total_orders
FROM orders o JOIN customers c ON o.customer_id = c.customer_id GROUP BY c.name
ORDER BY total_orders DESC LIMIT 1
"""
print(pd.read_sql(query, conn))

           name  total_orders
0  Rahul Sharma             2


In [75]:
query = "SELECT SUM(total_amount) AS total_revenue FROM orders"
print(pd.read_sql(query, conn))

   total_revenue
0        7994.96


# Task 3: Pandas Integration and Analysis

# Part A: SQL to Pandas



In [77]:
books_df = pd.read_sql("SELECT * FROM books", conn)
customers_df = pd.read_sql("SELECT * FROM customers", conn)
orders_df = pd.read_sql("SELECT * FROM orders", conn)

In [81]:
print(books_df.head(3))

   book_id                    title       author    price  stock_quantity
0        1       Python Programming   John Smith   599.99              25
1        2    Data Science Handbook     Jane Doe   899.50              15
2        3  Machine Learning Basics  Alan Turing  1299.00              10


In [83]:
print(customers_df.head(3))

   customer_id          name            email       city   join_date
0            1  Rahul Sharma  rahul@email.com     Mumbai  2024-01-15
1            2   Priya Patel  priya@email.com      Delhi  2024-01-20
2            3    Amit Kumar   amit@email.com  Bangalore  2024-02-01


In [85]:
print(orders_df.head(3))

   order_id  customer_id  book_id  quantity  order_date  total_amount
0         1            1        1         2  2024-03-01       1199.00
1         2            1        2         1  2024-03-02        899.50
2         3            2        1         1  2024-03-03        599.99


In [87]:
report = orders_df.merge(customers_df, on = 'customer_id').merge(books_df, on = 'book_id')

In [89]:
report = report[['order_id', 'name', 'city', 'title', 'quantity', 'total_amount']]
print(report)

   order_id          name       city                    title  quantity  \
0         1  Rahul Sharma     Mumbai       Python Programming         2   
1         2  Rahul Sharma     Mumbai    Data Science Handbook         1   
2         3   Priya Patel      Delhi       Python Programming         1   
3         4   Priya Patel      Delhi  Machine Learning Basics         1   
4         5    Amit Kumar  Bangalore           SQL Essentials         3   
5         6   Sneha Reddy  Hyderabad    Data Science Handbook         1   
6         7  Vikram Singh     Mumbai          Web Development         2   

   total_amount  
0       1199.00  
1        899.50  
2        599.99  
3       1299.00  
4       1499.97  
5        899.50  
6       1598.00  


In [93]:
print("Average Order Value:", report['total_amount'].mean())

Average Order Value: 1142.1371428571429


In [95]:
orders_by_city = report.groupby('city')['order_id'].count()

print("\nOrders by City")
print(orders_by_city)


Orders by City
city
Bangalore    1
Delhi        2
Hyderabad    1
Mumbai       3
Name: order_id, dtype: int64


In [97]:
popular_book = report.groupby('title')['quantity'].sum().sort_values(ascending=False)

print("\nMost Popular Book")
print(popular_book.head(1))


Most Popular Book
title
Python Programming    3
Name: quantity, dtype: int64


# Part B: Pandas to SQL



In [99]:
discounts_data = {
    'book_id': [1,2,3,4,5],
    'discount_percent': [10,15,5,20,12]
}

discounts_df = pd.DataFrame(discounts_data)

In [101]:
discounts_df.to_sql("discounts", conn, if_exists="replace", index=False)

print("✓ Discounts table created")

✓ Discounts table created


In [103]:
print(pd.read_sql("SELECT * FROM discounts", conn))

   book_id  discount_percent
0        1                10
1        2                15
2        3                 5
3        4                20
4        5                12


In [105]:
query = """
SELECT 
b.title,
b.price AS original_price,
d.discount_percent,
(b.price - (b.price * d.discount_percent / 100)) AS discounted_price
FROM books b
JOIN discounts d
ON b.book_id = d.book_id
"""

print(pd.read_sql(query, conn))

                     title  original_price  discount_percent  discounted_price
0       Python Programming          599.99                10           539.991
1    Data Science Handbook          899.50                15           764.575
2  Machine Learning Basics         1299.00                 5          1234.050
3           SQL Essentials          499.99                20           399.992
4          Web Development          799.00                12           703.120


In [107]:
conn.close()

print("✓ Database connection closed")

✓ Database connection closed
